In [2]:
# Aim: Initialize the Random Forest experiment with the canonical data and locked validation protocol.

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

RANDOM_STATE = 42

train_final = pd.read_csv("../data/train_final.csv")
test_final = pd.read_csv("../data/test_final.csv")

target = "total_sales"

rf_features = [
    "product_code",
    "product_weight_kg",
    "fat_content",
    "shelf_visibility",
    "product_price",
    "store_code",
    "store_age_years",
    "store_size",
    "store_location_tier",
    "store_format",
    "product_category_clean"
]

rf_categorical = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format"
]

rf_numeric = [
    "product_weight_kg",
    "shelf_visibility",
    "product_price",
    "store_age_years"
]

X = train_final[rf_features].copy()
y = train_final[target].copy()
X_test = test_final[rf_features].copy()

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Train shape:", X.shape)
print("Test shape:", X_test.shape)
print("Target shape:", y.shape)
print("Categorical features:", len(rf_categorical))
print("Numeric features:", len(rf_numeric))

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    print(
        f"Fold {fold}: "
        f"train={len(train_idx)}, "
        f"validation={len(val_idx)}"
    )

Train shape: (6818, 11)
Test shape: (1705, 11)
Target shape: (6818,)
Categorical features: 7
Numeric features: 4
Fold 1: train=5454, validation=1364
Fold 2: train=5454, validation=1364
Fold 3: train=5454, validation=1364
Fold 4: train=5455, validation=1363
Fold 5: train=5455, validation=1363


In [7]:
# Aim: Reproduce the original Random Forest baseline under the locked validation protocol.

from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

rf_preprocessor = ColumnTransformer([
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ))
        ]),
        rf_categorical
    ),
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]),
        rf_numeric
    )
])

rf_baseline_oof = np.zeros(len(y))
rf_baseline_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

    X_tr = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    X_tr_encoded = rf_preprocessor.fit_transform(X_tr)
    X_val_encoded = rf_preprocessor.transform(X_val)

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_tr_encoded, y_tr)

    pred = model.predict(X_val_encoded)

    rf_baseline_oof[val_idx] = pred

    rmse = np.sqrt(mean_squared_error(y_val, pred))
    rf_baseline_fold_rmse.append(rmse)

    print(f"Fold {fold}: {rmse:.6f}")

print(f"\nMean fold RMSE: {np.mean(rf_baseline_fold_rmse):.6f}")
print(f"OOF RMSE: {np.sqrt(mean_squared_error(y, rf_baseline_oof)):.6f}")

Fold 1: 1126.817185
Fold 2: 1082.201547
Fold 3: 1147.933435
Fold 4: 1161.498719
Fold 5: 1119.067128

Mean fold RMSE: 1127.503603
OOF RMSE: 1127.827713


In [9]:
# Aim: Measure Random Forest diversity relative to the frozen CatBoost OOF predictions.

from catboost import CatBoostRegressor

catboost_oof = np.zeros(len(y))
catboost_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

    X_tr = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    model = CatBoostRegressor(
        boosting_type="Ordered",
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        random_strength=1.0,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False
    )

    model.fit(
        X_tr,
        y_tr,
        cat_features=rf_categorical
    )

    pred = model.predict(X_val)

    catboost_oof[val_idx] = pred

    rmse = np.sqrt(mean_squared_error(y_val, pred))
    catboost_fold_rmse.append(rmse)

catboost_residual = y.to_numpy() - catboost_oof
rf_residual = y.to_numpy() - rf_baseline_oof

prediction_corr = np.corrcoef(
    catboost_oof,
    rf_baseline_oof
)[0, 1]

residual_corr = np.corrcoef(
    catboost_residual,
    rf_residual
)[0, 1]

print(f"CatBoost OOF RMSE: {np.sqrt(mean_squared_error(y, catboost_oof)):.6f}")
print(f"Random Forest OOF RMSE: {np.sqrt(mean_squared_error(y, rf_baseline_oof)):.6f}")
print(f"Prediction correlation: {prediction_corr:.6f}")
print(f"Residual correlation: {residual_corr:.6f}")

CatBoost OOF RMSE: 1075.086399
Random Forest OOF RMSE: 1127.827713
Prediction correlation: 0.962456
Residual correlation: 0.945473


In [10]:
# Aim: Determine whether Random Forest's residual diversity provides useful OOF ensemble improvement.

rf_blend_weights = np.arange(0.0, 0.51, 0.05)

for rf_weight in rf_blend_weights:
    cat_weight = 1.0 - rf_weight

    blend_oof = (
        cat_weight * catboost_oof
        + rf_weight * rf_baseline_oof
    )

    blend_rmse = np.sqrt(
        mean_squared_error(y, blend_oof)
    )

    print(
        f"CatBoost: {cat_weight:.2f} | "
        f"Random Forest: {rf_weight:.2f} | "
        f"OOF RMSE: {blend_rmse:.6f}"
    )

CatBoost: 1.00 | Random Forest: 0.00 | OOF RMSE: 1075.086399
CatBoost: 0.95 | Random Forest: 0.05 | OOF RMSE: 1074.766518
CatBoost: 0.90 | Random Forest: 0.10 | OOF RMSE: 1074.764745
CatBoost: 0.85 | Random Forest: 0.15 | OOF RMSE: 1075.081082
CatBoost: 0.80 | Random Forest: 0.20 | OOF RMSE: 1075.715249
CatBoost: 0.75 | Random Forest: 0.25 | OOF RMSE: 1076.666683
CatBoost: 0.70 | Random Forest: 0.30 | OOF RMSE: 1077.934545
CatBoost: 0.65 | Random Forest: 0.35 | OOF RMSE: 1079.517720
CatBoost: 0.60 | Random Forest: 0.40 | OOF RMSE: 1081.414823
CatBoost: 0.55 | Random Forest: 0.45 | OOF RMSE: 1083.624206
CatBoost: 0.50 | Random Forest: 0.50 | OOF RMSE: 1086.143961


In [11]:
# Aim: Locate the useful Random Forest contribution more precisely around the observed OOF blend optimum.

rf_fine_weights = np.arange(0.06, 0.15, 0.01)

best_weight = None
best_rmse = np.inf

for rf_weight in rf_fine_weights:
    cat_weight = 1.0 - rf_weight

    blend_oof = (
        cat_weight * catboost_oof
        + rf_weight * rf_baseline_oof
    )

    blend_rmse = np.sqrt(
        mean_squared_error(y, blend_oof)
    )

    print(
        f"CatBoost: {cat_weight:.2f} | "
        f"Random Forest: {rf_weight:.2f} | "
        f"OOF RMSE: {blend_rmse:.6f}"
    )

    if blend_rmse < best_rmse:
        best_rmse = blend_rmse
        best_weight = rf_weight

print(f"\nBest RF weight: {best_weight:.2f}")
print(f"Best OOF RMSE: {best_rmse:.6f}")
print(f"Improvement vs CatBoost: {1075.086399 - best_rmse:.6f}")

CatBoost: 0.94 | Random Forest: 0.06 | OOF RMSE: 1074.740710
CatBoost: 0.93 | Random Forest: 0.07 | OOF RMSE: 1074.727629
CatBoost: 0.92 | Random Forest: 0.08 | OOF RMSE: 1074.727275
CatBoost: 0.91 | Random Forest: 0.09 | OOF RMSE: 1074.739647
CatBoost: 0.90 | Random Forest: 0.10 | OOF RMSE: 1074.764745
CatBoost: 0.89 | Random Forest: 0.11 | OOF RMSE: 1074.802568
CatBoost: 0.88 | Random Forest: 0.12 | OOF RMSE: 1074.853116
CatBoost: 0.87 | Random Forest: 0.13 | OOF RMSE: 1074.916386
CatBoost: 0.86 | Random Forest: 0.14 | OOF RMSE: 1074.992375

Best RF weight: 0.08
Best OOF RMSE: 1074.727275
Improvement vs CatBoost: 0.359124


In [12]:
# Aim: Confirm whether the Random Forest blend improvement survives nested out-of-fold weight selection.

blend_candidates = [0.05, 0.075, 0.10, 0.125, 0.15]

nested_blend_oof = np.zeros(len(y))
nested_selected_weights = []
nested_fold_rmse = []

for outer_fold, (outer_train_idx, outer_val_idx) in enumerate(
    kf.split(X), start=1
):
    inner_kf = KFold(
        n_splits=4,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    inner_cat_oof = np.zeros(len(outer_train_idx))
    inner_rf_oof = np.zeros(len(outer_train_idx))

    X_outer_train = X.iloc[outer_train_idx]
    y_outer_train = y.iloc[outer_train_idx]

    # Generate inner OOF predictions for weight selection
    for inner_train_pos, inner_val_pos in inner_kf.split(X_outer_train):

        inner_train_idx = outer_train_idx[inner_train_pos]
        inner_val_idx = outer_train_idx[inner_val_pos]

        X_inner_tr = X.iloc[inner_train_idx]
        X_inner_val = X.iloc[inner_val_idx]

        y_inner_tr = y.iloc[inner_train_idx]
        y_inner_val = y.iloc[inner_val_idx]

        # CatBoost
        cat_model = CatBoostRegressor(
            boosting_type="Ordered",
            iterations=300,
            learning_rate=0.025,
            depth=4,
            l2_leaf_reg=1,
            random_strength=1.0,
            loss_function="RMSE",
            random_seed=RANDOM_STATE,
            verbose=False
        )

        cat_model.fit(
            X_inner_tr,
            y_inner_tr,
            cat_features=rf_categorical
        )

        inner_cat_oof[
            inner_val_pos
        ] = cat_model.predict(X_inner_val)

        # Random Forest preprocessing
        inner_preprocessor = ColumnTransformer([
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False
                    ))
                ]),
                rf_categorical
            ),
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median"))
                ]),
                rf_numeric
            )
        ])

        X_inner_tr_encoded = inner_preprocessor.fit_transform(X_inner_tr)
        X_inner_val_encoded = inner_preprocessor.transform(X_inner_val)

        rf_model = RandomForestRegressor(
            n_estimators=100,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        rf_model.fit(
            X_inner_tr_encoded,
            y_inner_tr
        )

        inner_rf_oof[
            inner_val_pos
        ] = rf_model.predict(X_inner_val_encoded)

    # Select blend weight using only inner OOF predictions
    inner_rmse = {}

    for rf_weight in blend_candidates:
        cat_weight = 1.0 - rf_weight

        inner_blend = (
            cat_weight * inner_cat_oof
            + rf_weight * inner_rf_oof
        )

        inner_rmse[rf_weight] = np.sqrt(
            mean_squared_error(
                y_outer_train,
                inner_blend
            )
        )

    selected_weight = min(
        inner_rmse,
        key=inner_rmse.get
    )

    nested_selected_weights.append(selected_weight)

    # Retrain both models on the complete outer training fold
    X_outer_val = X.iloc[outer_val_idx]
    y_outer_val = y.iloc[outer_val_idx]

    cat_model = CatBoostRegressor(
        boosting_type="Ordered",
        iterations=300,
        learning_rate=0.025,
        depth=4,
        l2_leaf_reg=1,
        random_strength=1.0,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False
    )

    cat_model.fit(
        X_outer_train,
        y_outer_train,
        cat_features=rf_categorical
    )

    outer_cat_pred = cat_model.predict(X_outer_val)

    outer_preprocessor = ColumnTransformer([
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ))
            ]),
            rf_categorical
        ),
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            rf_numeric
        )
    ])

    X_outer_train_encoded = outer_preprocessor.fit_transform(
        X_outer_train
    )
    X_outer_val_encoded = outer_preprocessor.transform(
        X_outer_val
    )

    rf_model = RandomForestRegressor(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    rf_model.fit(
        X_outer_train_encoded,
        y_outer_train
    )

    outer_rf_pred = rf_model.predict(
        X_outer_val_encoded
    )

    outer_blend = (
        (1.0 - selected_weight) * outer_cat_pred
        + selected_weight * outer_rf_pred
    )

    nested_blend_oof[outer_val_idx] = outer_blend

    rmse = np.sqrt(
        mean_squared_error(
            y_outer_val,
            outer_blend
        )
    )

    nested_fold_rmse.append(rmse)

    print(
        f"Outer fold {outer_fold}: "
        f"RF weight={selected_weight:.3f} | "
        f"RMSE={rmse:.6f}"
    )

nested_oof_rmse = np.sqrt(
    mean_squared_error(y, nested_blend_oof)
)

print("\nSelected RF weights:", nested_selected_weights)
print(f"Mean outer-fold RMSE: {np.mean(nested_fold_rmse):.6f}")
print(f"Nested OOF RMSE: {nested_oof_rmse:.6f}")
print(
    f"Improvement vs CatBoost: "
    f"{1075.086399 - nested_oof_rmse:.6f}"
)

Outer fold 1: RF weight=0.100 | RMSE=1065.745424
Outer fold 2: RF weight=0.075 | RMSE=1052.063788
Outer fold 3: RF weight=0.150 | RMSE=1087.450249
Outer fold 4: RF weight=0.075 | RMSE=1107.284025
Outer fold 5: RF weight=0.100 | RMSE=1061.821137

Selected RF weights: [0.1, 0.075, 0.15, 0.075, 0.1]
Mean outer-fold RMSE: 1074.872925
Nested OOF RMSE: 1075.054489
Improvement vs CatBoost: 0.031910


In [13]:
# Aim: Test whether modestly increasing Random Forest leaf size improves generalization.

rf_leaf2_oof = np.zeros(len(y))
rf_leaf2_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

    X_tr = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    rf_preprocessor = ColumnTransformer([
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ))
            ]),
            rf_categorical
        ),
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            rf_numeric
        )
    ])

    X_tr_encoded = rf_preprocessor.fit_transform(X_tr)
    X_val_encoded = rf_preprocessor.transform(X_val)

    model = RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_tr_encoded, y_tr)

    pred = model.predict(X_val_encoded)

    rf_leaf2_oof[val_idx] = pred

    rmse = np.sqrt(mean_squared_error(y_val, pred))
    rf_leaf2_fold_rmse.append(rmse)

    print(f"Fold {fold}: {rmse:.6f}")

print(f"\nMean fold RMSE: {np.mean(rf_leaf2_fold_rmse):.6f}")
print(f"OOF RMSE: {np.sqrt(mean_squared_error(y, rf_leaf2_oof)):.6f}")

Fold 1: 1120.217693
Fold 2: 1079.803852
Fold 3: 1142.556532
Fold 4: 1148.723606
Fold 5: 1112.866562

Mean fold RMSE: 1120.833649
OOF RMSE: 1121.098112


In [14]:
# Aim: Test whether stronger Random Forest leaf regularization improves generalization beyond min_samples_leaf=2.

rf_leaf4_oof = np.zeros(len(y))
rf_leaf4_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

    X_tr = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    rf_preprocessor = ColumnTransformer([
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ))
            ]),
            rf_categorical
        ),
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            rf_numeric
        )
    ])

    X_tr_encoded = rf_preprocessor.fit_transform(X_tr)
    X_val_encoded = rf_preprocessor.transform(X_val)

    model = RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=4,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_tr_encoded, y_tr)

    pred = model.predict(X_val_encoded)

    rf_leaf4_oof[val_idx] = pred

    rmse = np.sqrt(mean_squared_error(y_val, pred))
    rf_leaf4_fold_rmse.append(rmse)

    print(f"Fold {fold}: {rmse:.6f}")

print(f"\nMean fold RMSE: {np.mean(rf_leaf4_fold_rmse):.6f}")
print(f"OOF RMSE: {np.sqrt(mean_squared_error(y, rf_leaf4_oof)):.6f}")

Fold 1: 1099.714289
Fold 2: 1070.531325
Fold 3: 1134.582966
Fold 4: 1137.032009
Fold 5: 1106.255585

Mean fold RMSE: 1109.623235
OOF RMSE: 1109.891143


In [15]:
# Aim: Test whether stronger Random Forest leaf regularization continues to improve validation performance.

rf_leaf8_oof = np.zeros(len(y))
rf_leaf8_fold_rmse = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

    X_tr = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_tr = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    rf_preprocessor = ColumnTransformer([
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ))
            ]),
            rf_categorical
        ),
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")
                )
            ]),
            rf_numeric
        )
    ])

    X_tr_encoded = rf_preprocessor.fit_transform(X_tr)
    X_val_encoded = rf_preprocessor.transform(X_val)

    model = RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=8,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_tr_encoded, y_tr)

    pred = model.predict(X_val_encoded)

    rf_leaf8_oof[val_idx] = pred

    rmse = np.sqrt(mean_squared_error(y_val, pred))
    rf_leaf8_fold_rmse.append(rmse)

    print(f"Fold {fold}: {rmse:.6f}")

print(f"\nMean fold RMSE: {np.mean(rf_leaf8_fold_rmse):.6f}")
print(f"OOF RMSE: {np.sqrt(mean_squared_error(y, rf_leaf8_oof)):.6f}")

Fold 1: 1090.609816
Fold 2: 1069.498567
Fold 3: 1122.219031
Fold 4: 1125.100070
Fold 5: 1090.667646

Mean fold RMSE: 1099.619026
OOF RMSE: 1099.819262
